<a href="https://colab.research.google.com/github/jigme42/Biomass/blob/main/Chlorophyll_Indices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

These are the chlorophyll indices for Griffith University, Logan Campus. We first import the geemap library for use in visualization.

In [ ]:
# Installing and importing geemap
try:
    import geemap
except ModuleNotFoundError:
    if 'google.colab' in str(get_ipython()):
        print('geemap not found, installing via pip in Google Colab...')
        ! pip install geemap --quiet
        import geemap
    else:
        print('geemap not found, please install via conda in your environment')

geemap not found, installing via pip in Google Colab...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.0 MB/s eta 0:00:00


In [ ]:
! pip install geopandas

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 51.2 MB/s eta 0:00:00


In [ ]:
# Import the libraries
import geemap
import ee
import geopandas as gpd

Let's Authenticate and initialize GEE

In [ ]:
ee.Authenticate()

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=dXnr1ITi15x4TqTCkqobu9HDiLYSZ4_Xz0gUggT-Lm0&tc=AcaSWbIJiassuZ6ZeZoxfkbzDkMKTFKUqw59eoz4jEg&cc=q1W8P9258eMTchz_wtg2ydlIhM2mExDFroPRXt4J5i4

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AVHEtk5614y2yeDAiF6fRrSgybfX7qsxfixx9VReaSwi8Rk416C8MVRMZNs

Successfully saved authorization token.


In [ ]:
ee.Initialize()

We now import Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
Map = geemap.Map()
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(Togg…

In [ ]:
# Area of interest - the shapefile digitized on the GIS software
arboretum_shp = gpd.read_file("/content/drive/MyDrive/Griffith Logan/Logan_Campus/Logan_arb.shp")
aoi = geemap.geopandas_to_ee(arboretum_shp)
Map.addLayer(aoi, {}, "Area of Interest")
Map.centerObject(aoi, 17); # Center shp and Zoom 17X
Map

Map(bottom=754.0, center=[-27.665074088348057, 153.1557239585474], controls=(WidgetControl(options=['position'…

**Function for calculating chlorophyll indices**

In [ ]:
# Normalized Difference Vegetation Index
def getNDVI(image):
    # Compute the NDVI using an expression.
    NDVI = image.expression('((NIR - RED) / (NIR + RED))', {
            'NIR': image.select ('B8'),
            'RED': image.select ('B4')
        }).rename("NDVI")

    image = image.addBands(NDVI)

    return(image)

In [ ]:
# Modified Chlorophyll Absorption Ratio Index (MCARI)
def getMCARI(image):
    # Compute the MCARI using an expression.
    MCARI = image.expression('((NIR - ((B5 - B4) - 0.2 * (B5 - GREEN))) * (NIR / (B5 + B4 - (B5 * B4))))', {
            'NIR': image.select('B8'),
            'GREEN': image.select('B3'),
            'B4': image.select('B4'),
            'B5': image.select('B5')
        }).rename("MCARI")

    image = image.addBands(MCARI)

    return image


In [ ]:
# Normalized Difference Chlorophyll Index (NDCI)
def getNDCI(image):
    # Compute the NDCI using an expression.
    NDCI = image.expression('((B6 - B5) / (B6 + B5))', {
            'B5': image.select('B5'),
            'B6': image.select('B6')
        }).rename("NDCI")

    image = image.addBands(NDCI)

    return image

In [ ]:
# Red-Edge Chlorophyll Index (CIred-edge)
def getCIred_edge(image):
    # Compute the CIred-edge using an expression.
    CIred_edge = image.expression('((B7 / B5) - 1) / ((B7 / B5) + 1)', {
            'B5': image.select('B5'),
            'B7': image.select('B7')
        }).rename("CIred_edge")

    image = image.addBands(CIred_edge)

    return image


In [ ]:
# Green-Red Vegetation Index (GRVI)
def getGRVI(image):
    # Compute the GRVI using an expression.
    GRVI = image.expression('((B5 / B4) - 1) / ((B5 / B4) + 1)', {
            'B4': image.select('B4'),
            'B5': image.select('B5')
        }).rename("GRVI")

    image = image.addBands(GRVI)

    return image


In [ ]:
# Simple Ratio Chlorophyll Index (SRCI)
def getSRCI(image):
    # Compute the SRCI using an expression.
    SRCI = image.expression('(B5 / B4)', {
            'B4': image.select('B4'),
            'B5': image.select('B5')
        }).rename("SRCI")

    image = image.addBands(SRCI)

    return image


In [ ]:
# Chlorophyll Index Red-Edge Maximum Value (CIrem)
def getCIrem(image):
    # Compute the CIrem using an expression.
    CIrem = image.expression('((B7 - B5) / (B7 + B5 - B6))', {
            'B5': image.select('B5'),
            'B6': image.select('B6'),
            'B7': image.select('B7')
        }).rename("CIrem")

    image = image.addBands(CIrem)

    return image


In [ ]:
# Triangular Greenness Index (TGI)
def getTGI(image):
    # Compute the TGI using an expression.
    TGI = image.expression('(-0.5) * ((190 * (B2 - B3)) - (120 * (B1 - B2)) + 4)', {
            'B1': image.select('B1'),
            'B2': image.select('B2'),
            'B3': image.select('B3')
        }).rename("TGI")

    image = image.addBands(TGI)

    return image


In [ ]:
# Chlorophyll Absorption Ratio Index (CARI)
def getCARI(image):
    # Compute the CARI using an expression.
    CARI = image.expression('((B5 / B4) - (B6 / B4))', {
            'B4': image.select('B4'),
            'B5': image.select('B5'),
            'B6': image.select('B6')
        }).rename("CARI")

    image = image.addBands(CARI)

    return image


In [ ]:
# Structure Insensitive Pigment Index (SIPI)
def getSIPI(image):
    # Compute the SIPI using an expression.
    SIPI = image.expression('((B7 - B5) / (B7 + B5)) * (1 - (B5 / B4)) * (B6 / B4)', {
            'B4': image.select('B4'),
            'B5': image.select('B5'),
            'B6': image.select('B6'),
            'B7': image.select('B7')
        }).rename("SIPI")

    image = image.addBands(SIPI)

    return image


Functions for masking clouds, followed by the function for getting the image collections

In [ ]:
def maskS2clouds(image):
  qa = image.select('QA60')
  cloudBitMask = 1 << 10
  cirrusBitMask = 1 << 11
  mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))
  return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

In [ ]:
# Function for obtaining all the image collections while removing the redundancies
def process_s2_imagery(year, aoi):

  # Image collection for the specified year
  s2 = ee.ImageCollection('COPERNICUS/S2') \
      .filterDate(f'{year}-09-01', f'{year}-11-30').filterBounds(aoi) \
      .map(maskS2clouds).map(getNDVI).map(getMCARI).map(getNDCI).map(getCIred_edge).map(getGRVI).map(getSRCI).map(getCIrem).map(getTGI).map(getCARI).map(getSIPI)

  # Calculate median value of the imagery collection
  s2_median = s2.median()

  return s2, s2_median

We now define the color palettes for the different chlorophyll Indices

In [ ]:
# Define the color palette for the Chlorophyll Indices' layers
ndvi_colors = ['#FF0000', '#FFFF00', '#00FF00' ]  # green, yellow, red
mcari_colors = ['white', 'yellow', 'brown']
ndci_colors = ['#0000FF', '#00FFFF', '#FFFF00', '#FF0000']  # blue, cyan, yellow, red
CIred_edge_colors = ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718',
    '74A901', '66A000', '529400', '3E8601', '207401', '056201',
    '004C00', '023B01', '012E01', '011D01', '011301']
grvi_colors = ['#FF0000', '#FFFFFF', '#00FF00']  # red, white, green
srci_colors = ['blue','white', 'brown']
CL_rem_colors = ['red', 'white', 'blue']
tgi_colors = ['red', 'white', 'blue']
cari_colors = ['red', 'white', 'blue']
sipi_colors = ['red', 'white', 'blue']

In [ ]:
palette_ndvi = {"min":0, "max":1, 'palette':ndvi_colors}
palette_mcari = {"min": 0, "max": 1, 'palette':mcari_colors}
palette_ndci = {"min": 0, "max": 1, 'palette':ndci_colors}
palette_CIred_edge = {"min": 0, "max": 1, 'palette':CIred_edge_colors}
palette_grvi = {"min": 0, "max": 1, 'palette':grvi_colors}
palette_srci = {"min": 0, "max": 1, 'palette':srci_colors}
palette_CL_rem = {"min": 0, "max": 1, 'palette':CL_rem_colors}
palette_tgi = {"min": 0, "max": 1, 'palette':tgi_colors}
palette_cari = {"min": 0, "max": 1, 'palette':cari_colors}
palette_sipi = {"min": 0, "max": 1, 'palette':sipi_colors}

In [ ]:
def indices_export(indices, years, aoi):# Rename
    # Create the map
    Map = geemap.Map()
    Map.centerObject(aoi, 17)

    # Initialize empty lists to store the images for each index
    index_images = {index: [] for index in indices}
    palettes = {'NDVI': palette_ndvi, 'MCARI': palette_mcari, 'NDCI': palette_ndci, 'CIred_edge': palette_CIred_edge, 'GRVI': palette_grvi, 'SRCI': palette_srci, 'CIrem': palette_CL_rem, 'TGI': palette_tgi, 'CARI': palette_cari, 'SIPI': palette_sipi}

    # Process Sentinel-2 imagery for each year
    for year in years:
        # Process Sentinel-2 imagery for the year
        s2, s2_median = process_s2_imagery(year, aoi)

        # Add layers to the map and append the images to their respective lists
        for index in indices:
            Map.addLayer(s2_median.clip(aoi).select(index), palettes[index], f"{index}_{year}")
            index_images[index].append(s2_median.clip(aoi).select(index))

    Map.addLayerControl()

    # Add a color bar for each index layer
    for index in indices:
        Map.add_colorbar(palettes[index], label=index, layer_name=index, orientation="horizontal", transparent_bg=True)

    return Map, index_images


In [ ]:
# We call the function
indices = ['NDVI', 'MCARI', 'NDCI', 'CIred_edge', 'GRVI', 'SRCI', 'CIrem', 'TGI', 'CARI', 'SIPI']
years = range(2015, 2023)
Map, index_images = indices_export(indices, years, aoi)

In [ ]:
# We export the images. We need to iterate over the indices and their respective images
for index in indices:
    for i, year in enumerate(years):
        img = index_images[index][i]
        bounds_fc = aoi.map(lambda f: ee.Feature(f.geometry().bounds()))
        bounds = bounds_fc.geometry().getInfo()['coordinates'][0]
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=f"{index}_{year}",
            fileNamePrefix=f"{index}_{year}",
            folder='Chlorophyll_Logan_Indices', # Folder that will be created in the drive for saving these images
            scale=1,
            region=bounds
        )
        task.start()
